<a href="https://colab.research.google.com/github/Medhini-dotcom/seismic-wave-web-simulator/blob/main/Seismic_Wave_Simulator_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [21]:
%%html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>SeismoLab - Seismic Wave Simulator</title>
    <style>
        body { font-family: system-ui, -apple-system, sans-serif; text-align: center; background: #030712; margin: 0; padding: 12px; color: #f3f4f6; }
        .container { max-width: 800px; margin: auto; background: rgba(17, 24, 39, 0.95); padding: 16px; border-radius: 12px; border: 2px solid #06b6d4; box-shadow: 0 0 25px rgba(6, 182, 212, 0.15); }
        h2 { margin: 0 0 4px 0; color: #06b6d4; font-weight: 900; letter-spacing: -0.02em; text-shadow: 0 0 10px rgba(6, 182, 212, 0.5); font-size: 22px; }
        canvas { background: #010307; border: 2px solid #374151; border-radius: 8px; width: 100%; max-width: 760px; height: auto; display: block; margin: 0 auto; }
        .controls { margin-top: 12px; display: flex; justify-content: space-between; align-items: center; flex-wrap: wrap; gap: 12px; padding: 10px 16px; background: #0b0f19; border-radius: 8px; border: 1px solid #1f2937; }
        .control-group { display: flex; align-items: center; gap: 8px; }
        button { background: #06b6d4; color: #030712; border: none; padding: 8px 16px; border-radius: 6px; font-weight: 800; font-size: 13px; cursor: pointer; transition: all 0.15s ease; box-shadow: 0 0 8px rgba(6, 182, 212, 0.3); }
        button:hover { background: #ffffff; color: #000000; transform: translateY(-1px); }
        label { font-size: 11px; color: #9ca3af; font-weight: 700; text-transform: uppercase; letter-spacing: 0.05em; }
        input[type=range] { accent-color: #06b6d4; width: 130px; background: #374151; height: 6px; border-radius: 3px; cursor: pointer; }
        .hud { display: flex; justify-content: space-between; gap: 8px; margin-bottom: 10px; background: #090d16; padding: 10px 14px; border-radius: 8px; border: 1px solid #1f2937; text-align: left; }
        .hud-item { display: flex; flex-direction: column; flex: 1; }
        .hud-label { font-size: 9px; color: #9ca3af; text-transform: uppercase; font-weight: 800; letter-spacing: 0.05em; }
        .hud-val { font-size: 14px; color: #ffffff; font-weight: 800; font-family: monospace; }
        .sound-toggle { background: #111827; border: 1.5px solid #f43f5e; padding: 8px 12px; color: #f43f5e; font-weight: 800; cursor: pointer; border-radius: 6px; font-size: 12px; }
        .sound-toggle:not(.muted) { border-color: #10b981; color: #10b981; }
    </style>
</head>
<body>

<div class="container">
    <h2>SeismoLab Simulator</h2>
    <p style="color: #9ca3af; font-size: 12px; margin: 0 0 12px 0; font-weight: 500;">Vivid dual-phase wave engine & real-time synthesized audio feedback.</p>

    <div class="hud">
        <div class="hud-item">
            <span class="hud-label">Distance to Epicenter</span>
            <span class="hud-val" id="hudDist" style="color: #06b6d4;">120 km</span>
        </div>
        <div class="hud-item">
            <span class="hud-label">P / S Velocities</span>
            <span class="hud-val" style="color: #10b981;">6.0 km/s / 3.4 km/s</span>
        </div>
        <div class="hud-item">
            <span class="hud-label">S-P Interval (Δt)</span>
            <span class="hud-val" id="hudTimeDiff" style="color: #f43f5e;">--</span>
        </div>
    </div>

    <canvas id="simCanvas" width="760" height="300"></canvas>

    <div class="controls">
        <button id="playBtn">▶ Play</button>
        <button id="muteBtn" class="sound-toggle muted">🔊 Audio: OFF</button>
        <div class="control-group">
            <label for="distRange">Distance: <span id="distVal" style="color: #06b6d4; font-weight: 800;">120</span> km</label>
            <input type="range" id="distRange" min="40" max="280" value="120" step="10">
        </div>
    </div>
</div>

<script>
    const canvas = document.getElementById('simCanvas');
    const ctx = canvas.getContext('2d');
    let time = 0, playing = false, distance = 120, animationId;
    let audioCtx = null, noiseNode = null, lowpassFilter = null, gainNode = null;
    let audioEnabled = false;

    const vp = 3.5;
    const vs = 2.0;
    const fx = 100;
    const fy = 130;

    let seismogramData = new Array(350).fill(0);

    function initAudio() {
        try {
            audioCtx = new (window.AudioContext || window.webkitAudioContext)();
            const bufferSize = 2 * audioCtx.sampleRate;
            const noiseBuffer = audioCtx.createBuffer(1, bufferSize, audioCtx.sampleRate);
            const output = noiseBuffer.getChannelData(0);
            for (let i = 0; i < bufferSize; i++) {
                output[i] = Math.random() * 2 - 1;
            }
            noiseNode = audioCtx.createBufferSource();
            noiseNode.buffer = noiseBuffer;
            noiseNode.loop = true;
            lowpassFilter = audioCtx.createBiquadFilter();
            lowpassFilter.type = 'lowpass';
            lowpassFilter.frequency.value = 50;
            gainNode = audioCtx.createGain();
            gainNode.gain.setValueAtTime(0.01, audioCtx.currentTime);
            noiseNode.connect(lowpassFilter);
            lowpassFilter.connect(gainNode);
            gainNode.connect(audioCtx.destination);
            noiseNode.start();
        } catch(e) {
            console.error("Audio Context not supported.");
        }
    }

    function updateAudioSynth(hasP, hasS, decayP, decayS) {
        if (!audioCtx || !audioEnabled) return;
        if (audioCtx.state === 'suspended') {
            audioCtx.resume();
        }
        if (hasS) {
            lowpassFilter.frequency.setValueAtTime(55, audioCtx.currentTime);
            gainNode.gain.setValueAtTime(Math.max(0.01, 0.45 * decayS), audioCtx.currentTime);
        } else if (hasP) {
            lowpassFilter.frequency.setValueAtTime(120, audioCtx.currentTime);
            gainNode.gain.setValueAtTime(0.08, audioCtx.currentTime);
        } else {
            lowpassFilter.frequency.setValueAtTime(30, audioCtx.currentTime);
            gainNode.gain.setValueAtTime(0.01, audioCtx.currentTime);
        }
    }

    function draw() {
        ctx.clearRect(0, 0, canvas.width, canvas.height);
        const stationX = fx + (distance / 300) * 480;
        const stationY = 60;

        ctx.fillStyle = '#1e293b';
        ctx.fillRect(0, 0, canvas.width, 60);

        let gradient = ctx.createLinearGradient(0, 60, 0, 180);
        gradient.addColorStop(0, '#0b0f19');
        gradient.addColorStop(1, '#02040a');
        ctx.fillStyle = gradient;
        ctx.fillRect(0, 60, canvas.width, 120);

        ctx.strokeStyle = '#374151';
        ctx.lineWidth = 1.5;
        ctx.beginPath(); ctx.moveTo(0, 60); ctx.lineTo(canvas.width, 60); ctx.stroke();
        ctx.beginPath(); ctx.moveTo(0, 180); ctx.lineTo(canvas.width, 180); ctx.stroke();

        ctx.fillStyle = '#ffffff';
        ctx.font = 'bold 10px monospace';
        ctx.fillText("CRUST (6.0 km/s)", 15, 50);
        ctx.fillText("MANTLE", 15, 195);

        ctx.strokeStyle = 'rgba(255, 255, 255, 0.25)';
        ctx.setLineDash([4, 4]);
        ctx.beginPath();
        ctx.moveTo(fx, fy);
        ctx.lineTo(stationX, stationY);
        ctx.stroke();
        ctx.setLineDash([]);

        const distanceToStation = Math.hypot(stationX - fx, stationY - fy);
        const currentP_radius = time * vp;
        const currentS_radius = time * vs;

        if (currentS_radius > 0) {
            ctx.strokeStyle = '#f43f5e';
            ctx.lineWidth = 2.5;
            ctx.setLineDash([5, 3]);
            ctx.beginPath();
            ctx.arc(fx, fy, currentS_radius, 0, Math.PI * 2);
            ctx.stroke();
            ctx.setLineDash([]);
        }

        if (currentP_radius > 0) {
            ctx.strokeStyle = '#06b6d4';
            ctx.lineWidth = 3;
            ctx.beginPath();
            ctx.arc(fx, fy, currentP_radius, 0, Math.PI * 2);
            ctx.stroke();
        }

        ctx.fillStyle = '#ef4444';
        ctx.beginPath();
        ctx.arc(fx, fy, 8, 0, Math.PI * 2);
        ctx.fill();

        ctx.fillStyle = '#10b981';
        ctx.beginPath();
        ctx.moveTo(stationX, stationY);
        ctx.lineTo(stationX - 10, stationY - 15);
        ctx.lineTo(stationX + 10, stationY - 15);
        ctx.closePath();
        ctx.fill();

        ctx.fillStyle = '#010307';
        ctx.fillRect(15, 210, 730, 80);
        ctx.strokeStyle = '#374151';
        ctx.lineWidth = 1.5;
        ctx.strokeRect(15, 210, 730, 80);

        let currentNoise = 0;
        let hasPArrived = currentP_radius >= distanceToStation;
        let hasSArrived = currentS_radius >= distanceToStation;
        let decaySFactor = Math.exp(-0.007 * (time - distanceToStation/vs));

        if (playing) {
            if (hasSArrived) {
                currentNoise = (Math.sin(time * 1.5) + Math.cos(time * 0.7)) * 20 * decaySFactor;
            } else if (hasPArrived) {
                currentNoise = (Math.sin(time * 3.5)) * 5;
            } else {
                currentNoise = (Math.random() - 0.5) * 1.2;
            }
            seismogramData.push(currentNoise);
            if (seismogramData.length > 720) {
                seismogramData.shift();
            }
            updateAudioSynth(hasPArrived, hasSArrived, 1.0, decaySFactor);
        }

        ctx.strokeStyle = hasSArrived ? '#f43f5e' : (hasPArrived ? '#06b6d4' : '#10b981');
        ctx.lineWidth = 2.5;
        ctx.beginPath();
        for (let i = 0; i < seismogramData.length; i++) {
            let lineX = 15 + i;
            let lineY = 250 + seismogramData[i];
            if (i === 0) ctx.moveTo(lineX, lineY);
            else ctx.lineTo(lineX, lineY);
        }
        ctx.stroke();

        const pTime = (distance / 6.0).toFixed(1);
        const sTime = (distance / 3.4).toFixed(1);
        const sMinusP = (sTime - pTime).toFixed(1);

        document.getElementById('hudDist').textContent = `${distance} km`;
        document.getElementById('hudTimeDiff').textContent = `${sMinusP} s (P: ${pTime}s | S: ${sTime}s)`;
    }

    function loop() {
        if (playing) {
            time += 0.5;
            if (time > 220) {
                time = 0;
                seismogramData.fill(0);
            }
            draw();
            animationId = requestAnimationFrame(loop);
        }
    }

    document.getElementById('playBtn').addEventListener('click', () => {
        playing = !playing;
        document.getElementById('playBtn').textContent = playing ? '❚❚ Pause' : '▶ Play';
        if (playing) loop();
    });

    document.getElementById('muteBtn').addEventListener('click', (e) => {
        if (!audioCtx) {
            initAudio();
        }
        audioEnabled = !audioEnabled;
        if (audioEnabled) {
            e.target.textContent = "🔊 Audio: ON";
            e.target.classList.remove('muted');
            if (audioCtx.state === 'suspended') audioCtx.resume();
        } else {
            e.target.textContent = "🔇 Audio: OFF";
            e.target.classList.add('muted');
            if (gainNode) gainNode.gain.setValueAtTime(0, audioCtx.currentTime);
        }
    });

    document.getElementById('distRange').addEventListener('input', (e) => {
        distance = +e.target.value;
        document.getElementById('distVal').textContent = distance;
        time = 0;
        seismogramData.fill(0);
        draw();
    });

    draw();
</script>

</body>
</html>
